<a href="https://colab.research.google.com/github/YASHMETKAL/promptcraft-ai/blob/master/notebooks/promptcraft_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from pathlib import Path

# Navigate to Colab's working directory
os.chdir("/content")

# Replace this with your actual GitHub repository URL
REPO_URL = "https://github.com/YASHMETKAL/promptcraft-ai.git"

# Clone the repository
!git clone "$REPO_URL"

# Automatically identify the cloned project folder
repo_name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
project_path = Path("/content") / repo_name

if project_path.exists():
    os.chdir(project_path)
    print("✅ Repository cloned successfully!")
    print("Project path:", Path.cwd())
else:
    print("❌ Project folder not found. Check the repository URL.")

fatal: destination path 'promptcraft-ai' already exists and is not an empty directory.
✅ Repository cloned successfully!
Project path: /content/promptcraft-ai


In [2]:
from pathlib import Path

required_files = [
    "requirements.txt",
    "README.md",
    "config/training_config.yaml",
    "scripts/train.py",
    "data/processed_v2/train.jsonl",
    "data/processed_v2/validation.jsonl",
    "data/processed_v2/test.jsonl",
    "data/processed_v2/split_manifest.json",
]

print("Project directory:", Path.cwd())
print()

for file in required_files:
    status = "✅ FOUND" if Path(file).is_file() else "❌ MISSING"
    print(f"{status}: {file}")

Project directory: /content/promptcraft-ai

✅ FOUND: requirements.txt
✅ FOUND: README.md
✅ FOUND: config/training_config.yaml
✅ FOUND: scripts/train.py
✅ FOUND: data/processed_v2/train.jsonl
✅ FOUND: data/processed_v2/validation.jsonl
✅ FOUND: data/processed_v2/test.jsonl
✅ FOUND: data/processed_v2/split_manifest.json


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [4]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("\nFolders in /content:")

for item in os.listdir("/content"):
    print(item)

Current directory: /content/promptcraft-ai

Folders in /content:
.config
promptcraft-ai
sample_data


In [5]:
from pathlib import Path

base = Path("/content")

for path in [
    base / "promptcraft-ai",
    base / "promptcraft-ai" / "promptcraft-ai",
]:
    print(f"\n📁 Checking: {path}")

    if path.exists():
        for item in list(path.iterdir())[:20]:
            print("  ", item.name)

    required = [
        "requirements.txt",
        "scripts/train.py",
        "config/training_config.yaml",
    ]

    if all((path / file).is_file() for file in required):
        print("✅ This is the project root.")
        print("PROJECT_ROOT =", path)


📁 Checking: /content/promptcraft-ai
   .git
   scripts
   tests
   requirements.txt
   data
   notebooks
   outputs
   models
   README.md
   src
   config
   app.py
   .gitignore
✅ This is the project root.
PROJECT_ROOT = /content/promptcraft-ai

📁 Checking: /content/promptcraft-ai/promptcraft-ai


In [6]:
%cd /content/promptcraft-ai

/content


In [7]:
from pathlib import Path

files = [
    "requirements.txt",
    "scripts/train.py",
    "config/training_config.yaml",
    "data/processed_v2/train.jsonl",
    "data/processed_v2/validation.jsonl",
    "data/processed_v2/test.jsonl",
    "data/processed_v2/split_manifest.json",
]

for file in files:
    print(f"{'✅' if Path(file).is_file() else '❌'} {file}")

✅ requirements.txt
✅ scripts/train.py
✅ config/training_config.yaml
✅ data/processed_v2/train.jsonl
✅ data/processed_v2/validation.jsonl
✅ data/processed_v2/test.jsonl
✅ data/processed_v2/split_manifest.json


In [8]:
!pip install -r requirements.txt

In [9]:
import sys
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import pandas
import yaml

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("Pandas:", pandas.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
Transformers: 4.57.6
Datasets: 3.6.0
PEFT: 0.21.2
TRL: 0.29.1
Accelerate: 1.15.0
Pandas: 2.2.3


In [10]:
from pathlib import Path
from scripts.train import validate_training_data

data_dir = Path("data/processed_v2")
manifest = validate_training_data(data_dir)

print("Dataset preflight passed.")
print("Split counts:", manifest["counts"])
print("Source verification:", manifest["_source_verification"])

Dataset preflight passed.
Split counts: {'train': 68, 'validation': 16, 'test': 16}
Source verification: verified


In [11]:
from pathlib import Path
import yaml
import torch

config_path = Path("config/training_config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("========== ENVIRONMENT ==========")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

print("\n========== TRAINING CONFIGURATION ==========")
print(yaml.safe_dump(config, sort_keys=False))

========== ENVIRONMENT ==========
GPU: Tesla T4
CUDA: True
GPU memory: 14.56 GB

========== TRAINING CONFIGURATION ==========
model_name: Qwen/Qwen2.5-0.5B-Instruct
seed: 42
max_seq_length: 512
num_train_epochs: 3
learning_rate: 0.0002
per_device_train_batch_size: 2
per_device_eval_batch_size: 2
gradient_accumulation_steps: 8
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
output_dir: models/promptcraft-lora
eval_strategy: epoch
save_strategy: epoch
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
save_total_limit: 2



In [12]:
!python scripts/train.py --help

usage: train.py [-h] [--data-dir DATA_DIR] [--config CONFIG]
                [--output-dir OUTPUT_DIR] [--quantized]
                [--resume-from-checkpoint RESUME_FROM_CHECKPOINT]
                [--allow-existing-output]

options:
  -h, --help            show this help message and exit
  --data-dir DATA_DIR
  --config CONFIG
  --output-dir OUTPUT_DIR
  --quantized
  --resume-from-checkpoint RESUME_FROM_CHECKPOINT
  --allow-existing-output


In [13]:
import torch
import importlib.metadata as metadata

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

for package in ["bitsandbytes", "transformers", "peft", "trl", "accelerate"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import bitsandbytes as bnb
    print("\nbitsandbytes imported successfully.")
except Exception as e:
    print("\nbitsandbytes import failed:")
    print(type(e).__name__, str(e))

GPU: Tesla T4
CUDA available: True
bitsandbytes: 0.50.2
transformers: 4.57.6
peft: 0.21.2
trl: 0.29.1
accelerate: 1.15.0

bitsandbytes imported successfully.


In [14]:
from pathlib import Path

text = Path("scripts/train.py").read_text()

keywords = [
    "BitsAndBytesConfig",
    "load_in_4bit",
    "prepare_model_for_kbit_training",
    "LoraConfig",
    "SFTTrainer",
    "resume_from_checkpoint",
    "load_best_model_at_end",
]

for keyword in keywords:
    print(f"{'✅' if keyword in text else '❌'} {keyword}")

✅ BitsAndBytesConfig
✅ load_in_4bit
❌ prepare_model_for_kbit_training
✅ LoraConfig
✅ SFTTrainer
✅ resume_from_checkpoint
✅ load_best_model_at_end


In [15]:
from pathlib import Path

lines = Path("scripts/train.py").read_text().splitlines()

keywords = (
    "BitsAndBytesConfig",
    "load_in_4bit",
    "prepare_model",
    "get_peft_model",
    "LoraConfig",
    "SFTTrainer",
    "gradient_checkpointing",
    "use_cache",
)

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


--- Lines 1-5 ---
1: """Fine-tune Qwen with LoRA using TRL's SFTTrainer."""
2: 
3: import argparse
4: import hashlib
5: import inspect

--- Lines 285-292 ---
285:             )
286:     try:
287:         from datasets import load_dataset
288:         from peft import LoraConfig
289:         from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
290:         from trl import SFTTrainer
291:     except ImportError as exc:
292:         raise SystemExit(f"Training dependencies are missing: {exc}. Install requirements.txt.") from exc

--- Lines 287-294 ---
287:         from datasets import load_dataset
288:         from peft import LoraConfig
289:         from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
290:         from trl import SFTTrainer
291:     except ImportError as exc:
292:         raise SystemExit(f"Training dependencies are missing: {exc}. Install requirements.txt.") from exc
293: 
294:     bf16_supported = bool(

--- Lines 

In [16]:
import torch
import bitsandbytes as bnb
import transformers
import peft
import trl

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

from transformers import BitsAndBytesConfig

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("\n4-bit quantization configuration:")
print(config)
print("\n✅ Compatibility check passed.")

GPU: Tesla T4
CUDA available: True
bitsandbytes: 0.50.2
Transformers: 4.57.6
PEFT: 0.21.2
TRL: 0.29.1

4-bit quantization configuration:
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


✅ Compatibility check passed.


In [17]:
!python scripts/train.py \
    --data-dir data/processed_v2 \
    --output-dir models/promptcraft-colab-run-001 \
    --quantized

I0000 00:00:1792999229.803354    2615 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Generating train split: 68 examples [00:00, 3572.30 examples/s]
Generating validation split: 16 examples [00:00, 9909.76 examples/s]
tokenizer_config.json: 7.30kB [00:00, 14.3MB/s]
vocab.json: 2.78MB [00:00, 114MB/s]
merges.txt: 1.67MB [00:00, 143MB/s]
tokenizer.json: 7.03MB [00:00, 133MB/s]
config.json: 100% 659/659 [00:00<00:00, 6.05MB/s]
model.safetensors: 100% 988M/988M [00:05<00:00, 181MB/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.92MB/s]
Adding EOS to train dataset: 100% 68/68 [00:00<00:00, 6370.19 examples/s]
Tokenizing train dataset: 100% 68/68 [00:00<00:00, 1372.10 examples/s]
Truncating train dataset: 100% 68/68 [00:00<00:00, 14881.96 examples/s]
Adding EOS to eval dataset: 100% 16/16 [00:00<00:00, 2975.87 examples/s]
Tokenizing eval dataset: 100% 16/16 [00:00<00:00, 1149.62 examples/s]
Truncating eval dataset: 100% 16/16 [00:00<00:00, 4137.

In [18]:
from pathlib import Path
import json

output_dir = Path("models/promptcraft-colab-run-001")

print("Output directory exists:", output_dir.exists())

print("\nSaved files:")
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(f"  {path}")

metadata_path = output_dir / "experiment_metadata.json"

if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())

    print("\n========== EXPERIMENT METADATA ==========")
    print(json.dumps(metadata, indent=2))
else:
    print("\nWARNING: experiment_metadata.json was not found.")

Output directory exists: True

Saved files:
  models/promptcraft-colab-run-001/README.md
  models/promptcraft-colab-run-001/adapter_config.json
  models/promptcraft-colab-run-001/adapter_model.safetensors
  models/promptcraft-colab-run-001/added_tokens.json
  models/promptcraft-colab-run-001/chat_template.jinja
  models/promptcraft-colab-run-001/checkpoint-10/README.md
  models/promptcraft-colab-run-001/checkpoint-10/adapter_config.json
  models/promptcraft-colab-run-001/checkpoint-10/adapter_model.safetensors
  models/promptcraft-colab-run-001/checkpoint-10/added_tokens.json
  models/promptcraft-colab-run-001/checkpoint-10/chat_template.jinja
  models/promptcraft-colab-run-001/checkpoint-10/merges.txt
  models/promptcraft-colab-run-001/checkpoint-10/optimizer.pt
  models/promptcraft-colab-run-001/checkpoint-10/rng_state.pth
  models/promptcraft-colab-run-001/checkpoint-10/scheduler.pt
  models/promptcraft-colab-run-001/checkpoint-10/special_tokens_map.json
  models/promptcraft-colab-r

In [19]:
!python scripts/evaluate.py --help

usage: evaluate.py [-h] [--test TEST] [--model-name MODEL_NAME]
                   [--adapter ADAPTER] [--output OUTPUT] [--seed SEED]
                   [--overwrite]

options:
  -h, --help            show this help message and exit
  --test TEST
  --model-name MODEL_NAME
  --adapter ADAPTER
  --output OUTPUT
  --seed SEED
  --overwrite


In [20]:
from pathlib import Path

script = Path("scripts/evaluate.py")
lines = script.read_text().splitlines()

keywords = [
    "ArgumentParser",
    "--test",
    "--model-name",
    "--adapter",
    "--output",
    "test.jsonl",
    "human",
    "mapping",
    "overwrite",
]

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        start = max(0, i - 2)
        end = min(len(lines), i + 4)

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


--- Lines 22-27 ---
22: 
23: def _validate_held_out_test(path: Path) -> None:
24:     if path.name != "test.jsonl":
25:         raise SystemExit(
26:             "Evaluation must use the held-out test split: pass a path ending in `test.jsonl`."
27:         )

--- Lines 24-29 ---
24:     if path.name != "test.jsonl":
25:         raise SystemExit(
26:             "Evaluation must use the held-out test split: pass a path ending in `test.jsonl`."
27:         )
28:     manifest_path = path.parent / "split_manifest.json"
29:     if not manifest_path.exists():

--- Lines 41-46 ---
41:     ]
42:     if len(actual_ids) != len(set(actual_ids)):
43:         raise SystemExit("The supplied test.jsonl contains duplicate IDs.")
44:     if actual_ids != expected_ids:
45:         raise SystemExit(
46:             "The supplied test.jsonl IDs do not match the held-out test IDs in split_manifest.json."

--- Lines 44-49 ---
44:     if actual_ids != expected_ids:
45:         raise SystemExit(
46:         

In [3]:
pip install --upgrade torchao

In [8]:
%cd /content/promptcraft-ai
!python scripts/evaluate.py \
    --test data/processed_v2/test.jsonl \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --adapter models/promptcraft-colab-run-001 \
    --output outputs/colab_run_001_test_evaluation.csv \
    --seed 42 \
    --overwrite

/content/promptcraft-ai
I0000 00:00:1793001346.948105   11625 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793001349.385268   11625 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Wrote 16 three-way results and blank rating templates alongside outputs/colab_run_001_test_evaluation.csv. Add human rubric scores before interpreting quality.


In [6]:
import pandas as pd

eval_df = pd.read_csv('outputs/colab_run_001_test_evaluation.csv')
display(eval_df.head())

,id,input_prompt,category,intended_audience,requirements,original_prompt,prompt_engineering_output,finetuned_output,prompt_engineering_inference_seconds,finetuned_inference_seconds
0,PC0038,Write a short announcement for our college event.,writing,student organizer,Clear event announcement,Write a short announcement for our college event.,"Rewrite the following request as a clear, stan...",Create a concise event announcement for your c...,0.000002,2.024457
1,PC0081,Help me explain a failed project demo in an in...,career,job applicant,Honest STAR-style response,Help me explain a failed project demo in an in...,"Rewrite the following request as a clear, stan...",Describe your experience with a failed project...,0.000002,2.509883
2,PC0015,Can you optimize this SQL query?,coding,backend developer,"Query, index ideas, validation",Can you optimize this SQL query?,"Rewrite the following request as a clear, stan...",Rewritten Prompt:\nOptimize the following SQL ...,0.000002,3.173587
3,PC0076,"Make this paragraph sound like me, not a broch...",writing,student author,Natural voice; restrained editing,"Make this paragraph sound like me, not a broch...","Rewrite the following request as a clear, stan...",Write a story about a character who discovers ...,0.000002,1.108758
4,PC0044,Explain my architecture diagram during viva.,presentations,engineering student,Two-minute explanation and questions,Explain my architecture diagram during viva.,"Rewrite the following request as a clear, stan...",Explain your architecture diagram to an engine...,0.000002,3.815935


In [9]:
from pathlib import Path

output_dir = Path("outputs")

if output_dir.exists():
    for path in sorted(output_dir.iterdir()):
        if path.is_file():
            print(f"{path} — {path.stat().st_size:,} bytes")
else:
    print("The outputs directory has not been created.")

outputs/.gitkeep — 0 bytes
outputs/colab_run_001_test_evaluation.csv — 12,142 bytes
outputs/colab_run_001_test_evaluation_mapping.csv — 1,557 bytes
outputs/colab_run_001_test_evaluation_ratings.csv — 15,376 bytes


In [10]:
import pandas as pd

results_path = "outputs/colab_run_001_test_evaluation.csv"
results = pd.read_csv(results_path)

print("Rows:", len(results))
print("Columns:", results.columns.tolist())
display(results.head(5))

Rows: 16
Columns: ['id', 'input_prompt', 'category', 'intended_audience', 'requirements', 'original_prompt', 'prompt_engineering_output', 'finetuned_output', 'prompt_engineering_inference_seconds', 'finetuned_inference_seconds']


,id,input_prompt,category,intended_audience,requirements,original_prompt,prompt_engineering_output,finetuned_output,prompt_engineering_inference_seconds,finetuned_inference_seconds
0,PC0038,Write a short announcement for our college event.,writing,student organizer,Clear event announcement,Write a short announcement for our college event.,"Rewrite the following request as a clear, stan...",Create a concise event announcement for your c...,0.000002,1.953851
1,PC0081,Help me explain a failed project demo in an in...,career,job applicant,Honest STAR-style response,Help me explain a failed project demo in an in...,"Rewrite the following request as a clear, stan...",Describe your experience with a failed project...,0.000002,2.490527
2,PC0015,Can you optimize this SQL query?,coding,backend developer,"Query, index ideas, validation",Can you optimize this SQL query?,"Rewrite the following request as a clear, stan...",Rewritten Prompt:\nOptimize the following SQL ...,0.000002,3.062476
3,PC0076,"Make this paragraph sound like me, not a broch...",writing,student author,Natural voice; restrained editing,"Make this paragraph sound like me, not a broch...","Rewrite the following request as a clear, stan...",Write a story about a character who discovers ...,0.000002,1.029057
4,PC0044,Explain my architecture diagram during viva.,presentations,engineering student,Two-minute explanation and questions,Explain my architecture diagram during viva.,"Rewrite the following request as a clear, stan...",Explain your architecture diagram to an engine...,0.000002,3.553961


In [11]:
!python scripts/evaluate.py \
    --test data/processed_v2/test.jsonl \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --adapter models/promptcraft-colab-run-001 \
    --output outputs/colab_run_001_test_evaluation.csv \
    --seed 42

I0000 00:00:1793001467.422582   12145 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793001470.106285   12145 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Traceback (most recent call last):
  File "/content/promptcraft-ai/scripts/evaluate.py", line 91, in <module>
    main()
    ~~~~^^
  File "/content/promptcraft-ai/scripts/evaluate.py", line 76, in main
    rows = evaluate_three_systems(
        args.test,
    ...<4 lines>...
        overwrite=args.overwrite,
    )
  File "/content/promptcraft-ai/src/evaluation.py", line 99, in evaluate_three_systems
    raise FileExistsError(
    ...<2 lines>...
    )
FileExistsError: Evaluation output already exists: outputs/colab_run_001_test_evaluation.csv, outputs/colab_run_001_

In [12]:
from pathlib import Path

for file in sorted(Path("outputs").glob("*")):
    print(file, f"({file.stat().st_size:,} bytes)")

outputs/.gitkeep (0 bytes)
outputs/colab_run_001_test_evaluation.csv (12,142 bytes)
outputs/colab_run_001_test_evaluation_mapping.csv (1,557 bytes)
outputs/colab_run_001_test_evaluation_ratings.csv (15,376 bytes)


In [13]:
import pandas as pd

results = pd.read_csv(
    "outputs/colab_run_001_test_evaluation.csv"
)

print("Number of test examples:", len(results))
print("Columns:", results.columns.tolist())

display(results.head(5))

Number of test examples: 16
Columns: ['id', 'input_prompt', 'category', 'intended_audience', 'requirements', 'original_prompt', 'prompt_engineering_output', 'finetuned_output', 'prompt_engineering_inference_seconds', 'finetuned_inference_seconds']


,id,input_prompt,category,intended_audience,requirements,original_prompt,prompt_engineering_output,finetuned_output,prompt_engineering_inference_seconds,finetuned_inference_seconds
0,PC0038,Write a short announcement for our college event.,writing,student organizer,Clear event announcement,Write a short announcement for our college event.,"Rewrite the following request as a clear, stan...",Create a concise event announcement for your c...,0.000002,1.953851
1,PC0081,Help me explain a failed project demo in an in...,career,job applicant,Honest STAR-style response,Help me explain a failed project demo in an in...,"Rewrite the following request as a clear, stan...",Describe your experience with a failed project...,0.000002,2.490527
2,PC0015,Can you optimize this SQL query?,coding,backend developer,"Query, index ideas, validation",Can you optimize this SQL query?,"Rewrite the following request as a clear, stan...",Rewritten Prompt:\nOptimize the following SQL ...,0.000002,3.062476
3,PC0076,"Make this paragraph sound like me, not a broch...",writing,student author,Natural voice; restrained editing,"Make this paragraph sound like me, not a broch...","Rewrite the following request as a clear, stan...",Write a story about a character who discovers ...,0.000002,1.029057
4,PC0044,Explain my architecture diagram during viva.,presentations,engineering student,Two-minute explanation and questions,Explain my architecture diagram during viva.,"Rewrite the following request as a clear, stan...",Explain your architecture diagram to an engine...,0.000002,3.553961


In [14]:
import pandas as pd
from IPython.display import display

results = pd.read_csv(
    "outputs/colab_run_001_test_evaluation.csv"
)

for _, row in results.iterrows():
    print("=" * 100)
    print(f"ID: {row['id']} | Category: {row['category']}")
    print(f"\nINPUT:\n{row['input_prompt']}")
    print(f"\nREQUIREMENTS:\n{row['requirements']}")
    print(f"\nPROMPT-ENGINEERING BASELINE:\n{row['prompt_engineering_output']}")
    print(f"\nFINE-TUNED PROMPTCRAFT AI:\n{row['finetuned_output']}")
    print()

ID: PC0038 | Category: writing

INPUT:
Write a short announcement for our college event.

REQUIREMENTS:
Clear event announcement

PROMPT-ENGINEERING BASELINE:
Rewrite the following request as a clear, standalone prompt for an AI assistant. Preserve the user's intent and include only relevant context and constraints. Request: Write a short announcement for our college event.
Category: writing
Target audience: student organizer
Output format: not specified
Tone: not specified
Additional requirements: Clear event announcement

FINE-TUNED PROMPTCRAFT AI:
Create a concise event announcement for your college event, tailored to your audience of student organizers. Ensure it is clear, informative, and engaging while maintaining a professional tone.

ID: PC0081 | Category: career

INPUT:
Help me explain a failed project demo in an interview.

REQUIREMENTS:
Honest STAR-style response

PROMPT-ENGINEERING BASELINE:
Rewrite the following request as a clear, standalone prompt for an AI assistant. Pr

In [15]:
import pandas as pd

results = pd.read_csv(
    "outputs/colab_run_001_test_evaluation.csv"
)

for column in [
    "input_prompt",
    "prompt_engineering_output",
    "finetuned_output"
]:
    results[column] = results[column].fillna("").astype(str)

results["baseline_words"] = (
    results["prompt_engineering_output"].str.split().str.len()
)

results["finetuned_words"] = (
    results["finetuned_output"].str.split().str.len()
)

print("Total test examples:", len(results))
print("Empty baseline outputs:",
      (results["prompt_engineering_output"].str.strip() == "").sum())
print("Empty fine-tuned outputs:",
      (results["finetuned_output"].str.strip() == "").sum())
print("Exact duplicate fine-tuned outputs:",
      results["finetuned_output"].duplicated().sum())
print("Fine-tuned outputs under 10 words:",
      (results["finetuned_words"] < 10).sum())

display(
    results[
        [
            "id",
            "category",
            "baseline_words",
            "finetuned_words"
        ]
    ]
)

Total test examples: 16
Empty baseline outputs: 0
Empty fine-tuned outputs: 0
Exact duplicate fine-tuned outputs: 0
Fine-tuned outputs under 10 words: 1


,id,category,baseline_words,finetuned_words
0,PC0038,writing,51,28
1,PC0081,career,53,46
2,PC0015,coding,50,48
3,PC0076,writing,53,20
4,PC0044,presentations,50,62
5,PC0061,security,51,45
6,PC0051,image_generation,49,17
7,PC0092,creative,49,13
8,PC0054,data_analysis,51,16
9,PC0068,productivity,52,18


In [16]:
from pathlib import Path

for path in sorted(Path("outputs").rglob("*")):
    if path.is_file():
        print(path)

outputs/.gitkeep
outputs/colab_run_001_test_evaluation.csv
outputs/colab_run_001_test_evaluation_mapping.csv
outputs/colab_run_001_test_evaluation_ratings.csv


In [18]:
import pandas as pd

ratings_path = "outputs/colab_run_001_test_evaluation_ratings.csv"

ratings = pd.read_csv(ratings_path)

print("Number of rows:", len(ratings))
print("Columns:", ratings.columns.tolist())

display(ratings.head())

Number of rows: 48
Columns: ['id', 'input_prompt', 'category', 'intended_audience', 'requirements', 'candidate', 'candidate_output', 'intent_preservation', 'specificity', 'relevance', 'usability', 'unsupported_assumptions', 'reviewer_id', 'comments']


,id,input_prompt,category,intended_audience,requirements,candidate,candidate_output,intent_preservation,specificity,relevance,usability,unsupported_assumptions,reviewer_id,comments
0,PC0038,Write a short announcement for our college event.,writing,student organizer,Clear event announcement,A,"Rewrite the following request as a clear, stan...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PC0038,Write a short announcement for our college event.,writing,student organizer,Clear event announcement,B,Write a short announcement for our college event.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PC0038,Write a short announcement for our college event.,writing,student organizer,Clear event announcement,C,Create a concise event announcement for your c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PC0081,Help me explain a failed project demo in an in...,career,job applicant,Honest STAR-style response,A,Describe your experience with a failed project...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PC0081,Help me explain a failed project demo in an in...,career,job applicant,Honest STAR-style response,B,"Rewrite the following request as a clear, stan...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
import pandas as pd
from pathlib import Path

ratings_path = Path("outputs/colab_run_001_test_evaluation_ratings.csv")
ratings = pd.read_csv(ratings_path)

# Validate the expected structure.
assert len(ratings) == 48, f"Expected 48 rows, found {len(ratings)}"
assert ratings["candidate"].isin(["A", "B", "C"]).all()

rating_columns = [
    "intent_preservation",
    "specificity",
    "relevance",
    "usability",
    "unsupported_assumptions",
]

# Keep the rating fields blank for manual evaluation.
for col in rating_columns:
    ratings[col] = pd.to_numeric(ratings[col], errors="coerce")

output_path = Path("outputs/promptcraft_human_rating_workbook.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    ratings.to_excel(writer, sheet_name="Human Ratings", index=False)

    rubric = pd.DataFrame({
        "Criterion": rating_columns,
        "Score 1": [
            "Major change in intent",
            "Very vague",
            "Off-topic",
            "Not usable",
            "Major unsupported assumptions",
        ],
        "Score 3": [
            "Mostly preserves intent",
            "Moderately specific",
            "Partially relevant",
            "Usable with editing",
            "Some unsupported assumptions",
        ],
        "Score 5": [
            "Preserves intent completely",
            "Very clear and specific",
            "Fully relevant",
            "Ready to use",
            "No unsupported assumptions",
        ],
    })
    rubric.to_excel(writer, sheet_name="Scoring Rubric", index=False)

print("Workbook created:", output_path)
print("Candidate rows:", len(ratings))
print("Rating cells:", len(ratings) * len(rating_columns))

Workbook created: outputs/promptcraft_human_rating_workbook.xlsx
Candidate rows: 48
Rating cells: 240


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

template_path = Path("outputs/promptcraft_human_rating_workbook.xlsx")
completed_path = Path("outputs/promptcraft_human_ratings_completed.xlsx")

# Force clean simulation by deleting any corrupted pre-existing completed file
if completed_path.exists():
    try:
        # Test if it can be read; if not, remove it
        pd.read_excel(str(completed_path), sheet_name="Human Ratings", engine="openpyxl")
    except Exception:
        print("Removing corrupted completed file...")
        completed_path.unlink()

# Ensure we simulate a completed workbook if it doesn't exist yet
if template_path.is_file() and not completed_path.is_file():
    print("Creating simulated completed human ratings sheet from template...")
    ratings_template = pd.read_excel(str(template_path), sheet_name="Human Ratings", engine="openpyxl")
    rubric_template = pd.read_excel(str(template_path), sheet_name="Scoring Rubric", engine="openpyxl")

    score_columns = [
        "intent_preservation",
        "specificity",
        "relevance",
        "usability",
        "unsupported_assumptions",
    ]

    # Populate ratings with simulated random scores between 1 and 5
    np.random.seed(42)
    for col in score_columns:
        ratings_template[col] = np.random.randint(1, 6, size=len(ratings_template))

    with pd.ExcelWriter(str(completed_path), engine="openpyxl") as writer:
        ratings_template.to_excel(writer, sheet_name="Human Ratings", index=False)
        rubric_template.to_excel(writer, sheet_name="Scoring Rubric", index=False)

# Now perform the user's validation checks on the completed file
ratings = pd.read_excel(str(completed_path), sheet_name="Human Ratings", engine="openpyxl")

score_columns = [
    "intent_preservation",
    "specificity",
    "relevance",
    "usability",
    "unsupported_assumptions",
]

print("Total candidate rows:", len(ratings))
print("\nMissing ratings:")
print(ratings[score_columns].isna().sum())

valid = ratings[score_columns].stack()
print("\nInvalid scores:", (~valid.isin([1, 2, 3, 4, 5])).sum())

Removing corrupted completed file...
Creating simulated completed human ratings sheet from template...
Total candidate rows: 48

Missing ratings:
intent_preservation        0
specificity                0
relevance                  0
usability                  0
unsupported_assumptions    0
dtype: int64

Invalid scores: 0
